# Imports

In [1]:
import gymnasium as gym
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
import matplotlib.pyplot as plt

# Helpers

In [2]:
def softmax(s, a, theta):
    phi = np.zeros(32)
    phi[a*8:a*8+8] = s
    numerator = np.exp(phi.T @ theta)
    
    denominator = 0
    for a2 in range(4):
        phi2 = np.zeros(32)
        phi2[a2*8:a2*8+8] = s
        denominator += np.exp(phi2.T @ theta)
    
    pi = numerator / denominator
    return pi

In [3]:
def gradient(s, a, theta):
    phi = np.zeros(32)
    phi[a*8:a*8+8] = s

    expected = np.zeros(32)
    for a2 in range(4):
        phi2 = np.zeros(32)
        phi2[a2*8:a2*8+8] = s
        expected += phi2 * softmax(s, a2, theta)
    
    return phi - expected

In [4]:
def compute_returns(rewards, gamma):
    returns = []
    v = 0
    for r in reversed(rewards):
        v = r + gamma * v
        returns.insert(0, v)
    return returns

Main Reinforce Function

In [8]:
def reinforce(env, theta, alpha, gamma, n_episodes):
    for episode in range(n_episodes):
        if episode%50==0:
            print(episode)
        
        states, actions, rewards = [], [], []
        
        # env.reset() returns (state, info) 
        state, info = env.reset()
        done = False
        
        # generate full episode
        while not done:
            # sample action:
            probs = np.array([softmax(state, a, theta) for a in range(4)])
            action = np.random.choice(4, p=probs)
            next_state, reward, terminated, truncated, info = env.step(action)
            done = terminated or truncated
            # store state, action, reward in lists
            rewards.append(reward)
            states.append(state)
            actions.append(action)
            state = next_state
        returns = compute_returns(rewards, gamma)
        for t in range(len(states)):
            theta += alpha * gradient(states[t], actions[t], theta) * returns[t]
    return theta

# Testing

In [9]:
env = gym.make("LunarLander-v3")
theta = np.zeros(32)
alpha = 1e-4
gamma = 0.99
n_episodes = 2000

theta = reinforce(env, theta, alpha, gamma, n_episodes)

0
50
100
150
200
250
300
350
400
450
500
550
600
650
700
750
800
850
900
950
1000
1050
1100
1150
1200
1250
1300
1350
1400
1450
1500
1550
1600
1650
1700
1750
1800
1850
1900
1950


In [19]:
env_render = gym.make("LunarLander-v3", render_mode="human")
state, info = env_render.reset()
done = False

while not done:
    probs = np.array([softmax(state, a, theta) for a in range(4)])
    action = np.random.choice(4, p=probs)
    state, reward, terminated, truncated, info = env_render.step(action)
    done = terminated or truncated

env_render.close()